In [ ]:
# training.ipynb
# Stage 2 of 2: the full sweep via the supervisor. Run AFTER
# prepare_training.ipynb (which builds the H5 AND smoke-validates the training
# pipeline), so this notebook is just the multi-hour run + resume.
import subprocess

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG = "/workspace/stable_query_latent_logs/pipeline.log"

# GPUs to run across on THIS machine. Default: every visible GPU, one worker per
# card, all training a different combo at once (shared in-process grid).
# Override manually, e.g. GPUS = "0" (single) or GPUS = "0,1".
def _detect_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=5).stdout.strip()
        ids = [l.strip() for l in out.splitlines() if l.strip()]
        return ",".join(ids) if ids else "0"
    except Exception:
        return "0"

GPUS = _detect_gpus()

# ---- multi-MACHINE sharding (only if you run 2+ VMs on the SAME sweep) ----
# The grid is split into DISJOINT slices "i/N". Each VM runs a different slice and
# writes to ITS OWN out_dir, so their ledgers never contend on the network FS.
#   1 VM  : SHARD = "0/1"      (the whole sweep -- default)
#   2 VMs : VM-A SHARD="0/2", VM-B SHARD="1/2"
#   3 VMs : "0/3", "1/3", "2/3"   ... etc.
# Within a VM, GPUS still parallelises that VM's slice across its own cards.
SHARD = "0/1"

OUT_DIR_BASE = "VICReg_review/heads/cloud_full_sweep_a100"   # matches sweep.yaml
_i, _n = (int(x) for x in SHARD.split("/"))
OUT_DIR = OUT_DIR_BASE if _n == 1 else f"{OUT_DIR_BASE}_shard{_i}of{_n}"

print('repo :', REPO)
print('log  :', LOG)
print('gpus :', GPUS, '(one worker per GPU on this machine)')
print('shard:', SHARD, '-> out_dir', OUT_DIR)

In [ ]:
# Clone or pull before every run.
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git remote -v
!git pull origin main
!git status --short
!git rev-parse HEAD


In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor.
import sys, subprocess, threading, time, psutil
from pathlib import Path

if REPO not in sys.path:
    sys.path.insert(0, REPO)
try:
    # Same accounting the sweep planner budgets against (torch-free helper), so
    # what you see here == what the supervisor thinks is free.
    from VICReg_review.mem_budget import memory_breakdown as _mem_breakdown
except Exception:
    _mem_breakdown = None

stop = False

def _inline_breakdown():
    # Fallback if the repo isn't importable: read the cgroup directly.
    def _ri(p):
        try:
            t = Path(p).read_text().strip()
            return None if t == "max" else int(t)
        except Exception:
            return None
    limit = _ri("/sys/fs/cgroup/memory.max") or _ri("/sys/fs/cgroup/memory/memory.limit_in_bytes")
    stat = {}
    for p in ("/sys/fs/cgroup/memory.stat", "/sys/fs/cgroup/memory/memory.stat"):
        try:
            for line in Path(p).read_text().splitlines():
                k, _, v = line.partition(" ")
                if v.strip().isdigit():
                    stat[k] = int(v)
            if stat:
                break
        except Exception:
            pass
    if limit and stat and limit < 10**18:
        real = stat.get("anon", stat.get("rss", 0)) + stat.get("shmem", 0) + stat.get("unevictable", 0)
        cache = stat.get("file", stat.get("cache", 0))
        return {"source": "cgroup", "limit": float(limit), "real": float(real),
                "cache": float(cache), "usable": float(max(0, limit - real))}
    vm = psutil.virtual_memory()
    return {"source": "host", "limit": float(vm.total), "real": float(vm.total - vm.available),
            "cache": float(getattr(vm, "cached", 0)), "usable": float(vm.available)}

def get_memory_status():
    b = None
    if _mem_breakdown is not None:
        try:
            b = _mem_breakdown()
        except Exception:
            b = None
    if not b or not b.get("limit"):
        b = _inline_breakdown()
    lim = b["limit"] or 1.0
    return (b["real"] / lim * 100, b["real"] / 1024**3, b["cache"] / 1024**3,
            b["usable"] / 1024**3, b["limit"] / 1024**3, b["source"])

def get_gpu_status():
    # One entry per GPU so a multi-lane (multi-GPU) run shows every card.
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        cards = []
        for line in out.splitlines():
            p = [x.strip() for x in line.split(",")]
            cards.append(f"g{p[0]}:{float(p[1]):.0f}% {float(p[2])/1024:.1f}/{float(p[3])/1024:.1f}G")
        return " | ".join(cards) if cards else "n/a"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_real, ram_cache, ram_usable, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] real {ram_real:.0f}/{ram_total:.0f} GiB "
            f"(usable {ram_usable:.0f}, cache {ram_cache:.0f}) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()

In [ ]:
# Stage the embedding H5 to FAST LOCAL disk (overlay/NVMe) if it fits; else fall
# back to the network volume (/workspace = MooseFS, ~256MB/s read cap). Only the
# big READ (training) is redirected via --h5; checkpoints/ledger keep writing to
# out_dir on /workspace. Idempotent: skips if prepare_training already staged it
# this session. Non-fatal: any failure -> H5_PATH stays the /workspace H5.
import os, threading, time

WORKSPACE_H5 = os.path.join(REPO, "game_review_data/embedding_h5.h5")
LOCAL_H5 = "/root/data/embedding_h5.h5"
H5_PATH = WORKSPACE_H5   # safe default / fallback


def _free_bytes(path):
    st = os.statvfs(path)
    return st.f_bavail * st.f_frsize


def parallel_copy(src, dst, workers=8, chunk=64 << 20):
    """Thread-parallel byte-range copy (file I/O releases the GIL, so threads
    overlap the network reads; no fork/CUDA hazard from the notebook kernel)."""
    size = os.path.getsize(src)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "wb") as f:
        f.truncate(size)                       # preallocate
    bounds = [(i * size // workers, (i + 1) * size // workers) for i in range(workers)]
    errors = []

    def copy_range(start, end):
        sfd = os.open(src, os.O_RDONLY)
        dfd = os.open(dst, os.O_WRONLY)
        try:
            off = start
            while off < end:
                data = os.pread(sfd, min(chunk, end - off), off)
                if not data:
                    break
                os.pwrite(dfd, data, off)
                off += len(data)
        except BaseException as exc:            # noqa: BLE001
            errors.append(exc)
        finally:
            os.close(sfd)
            os.close(dfd)

    threads = [threading.Thread(target=copy_range, args=b) for b in bounds]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    if errors:
        raise errors[0]
    if os.path.getsize(dst) != size:
        raise IOError("size mismatch after copy")


try:
    src_size = os.path.getsize(WORKSPACE_H5)
    if os.path.exists(LOCAL_H5) and os.path.getsize(LOCAL_H5) == src_size:
        H5_PATH = LOCAL_H5
        print(f"local H5 already staged: {LOCAL_H5} ({src_size / 2**30:.1f} GiB)")
    else:
        free = _free_bytes("/")
        need = src_size + (5 << 30)            # 5 GiB headroom
        print(f"H5={src_size / 2**30:.1f}GiB  local('/') free={free / 2**30:.1f}GiB  need~{need / 2**30:.1f}GiB")
        if free >= need:
            t0 = time.time()
            parallel_copy(WORKSPACE_H5, LOCAL_H5, workers=8)
            dt = max(time.time() - t0, 1e-6)
            H5_PATH = LOCAL_H5
            print(f"staged -> {LOCAL_H5} in {dt:.0f}s ({src_size / 2**30 / dt:.2f} GiB/s)")
        else:
            print("not enough local space -> using /workspace H5 (expand Container Disk to stage locally)")
except BaseException as exc:
    print(f"staging failed ({type(exc).__name__}: {exc}) -> using /workspace H5")
    try:
        if os.path.exists(LOCAL_H5) and os.path.getsize(LOCAL_H5) != os.path.getsize(WORKSPACE_H5):
            os.remove(LOCAL_H5)
    except OSError:
        pass
    H5_PATH = WORKSPACE_H5

print("H5_PATH =", H5_PATH)


In [ ]:
# (Optional) Inspect the memory plan before the full run. Calibrates (C,R) on
# pseudo batches, then prints per combo the chosen backward-mode + stem chunk
# size. Safe to skip -- the supervisor calibrates + plans automatically.
# Uses the staged local H5 (H5_PATH) and this shard's OUT_DIR for calib.json.
!python -u VICReg_review/oom_proxy.py \
  --h5 {H5_PATH} \
  --calib-out {OUT_DIR}/calib.json --measure
!python -u -m VICReg_review.sweep.planner \
  --config VICReg_review/sweep/sweep.yaml \
  --h5 {H5_PATH} \
  --calib-json {OUT_DIR}/calib.json

In [ ]:
# Train the full sweep via the NEW supervisor (clean redesign; replaces sweep_cloud).
# Reads VICReg_review/sweep/sweep.yaml; --h5 redirects the big READ to the staged
# local copy (H5_PATH) while checkpoints/ledger write to OUT_DIR on /workspace.
# --gpus runs ONE worker per GPU on THIS machine, each training a different combo,
# all sharing one in-process grid so every combo (in this shard) is trained once.
# --shard/--out-dir carve out this VM's disjoint slice + its own state dir, so a
# second VM (SHARD "1/2") runs the other half without ever touching this ledger.
# Chunked stem -> FULL data, NO sentence cap. Crash-isolated PER LANE (shrink chunk
# on OOM / RAM-downgrade on SIGKILL, restart just that worker, resume from ledger).
# Host RAM budget + H5-read procs auto-scale to this machine's cores / lane-count.
# Resume: re-run this cell -- combos verified done (vs checkpoint) are skipped.
# State: {OUT_DIR}/{ledger.jsonl, calib.json, sweep_jobs/gpu*/}
import shlex

_cmd = ["python", "-u", "VICReg_review/sweep/supervisor.py",
        "--config", "VICReg_review/sweep/sweep.yaml",
        "--h5", H5_PATH, "--gpus", GPUS, "--out-dir", OUT_DIR,
        "--logout-address", LOG]
if SHARD != "0/1":
    _cmd += ["--shard", SHARD]
print("run:", " ".join(shlex.quote(c) for c in _cmd), flush=True)
!{" ".join(shlex.quote(c) for c in _cmd)}

In [ ]:
# Training done. Stop the monitor and print the ledger summary.
# Probe draining + final eval + artifact collection live in eval.ipynb
# (run it next, on this same pod, after training finishes).
# NOTE: reads THIS shard's OUT_DIR. If you sharded across VMs, each VM has its own
# ledger under its OUT_DIR; merge the shard dirs at eval time.
stop = True

import json
from collections import Counter
from pathlib import Path

out_dir = Path(REPO, OUT_DIR)
ledger = out_dir / 'ledger.jsonl'
queue_dir = out_dir / 'probe_queue'

if ledger.exists():
    latest = {}
    for line in ledger.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if rec.get('combo_id'):
            latest[rec['combo_id']] = rec
    counts = Counter(r.get('status', '?') for r in latest.values())
    print(f'ledger: {dict(counts)} ({len(latest)} combos)  [shard {SHARD} @ {OUT_DIR}]')
    failed = [c for c, r in latest.items() if r.get('status') == 'failed']
    if failed:
        print(f'failed combos ({len(failed)}):', failed)
else:
    print('no ledger yet:', ledger)

pending = sorted(queue_dir.glob('*.json')) if queue_dir.exists() else []
print(f'probe jobs queued (drain these in eval.ipynb): {len(pending)}')
print('Next: run Pod/eval.ipynb on this pod to drain probes, run final eval, and archive.')